# Biased-generalization analysis (DiffiT 256²)

Interactive version of `experiments/analyze_sample_split.py`. Loads matching
checkpoints from the two split-A / split-B training runs, computes the
sample-split cosine distance with matched noise, reads the already-logged
`Loss/test` from each run's `stats.jsonl`, and plots the dual-axis
biased-generalization figure with Plotly.

**Shape of the plot:**
- Left axis (blue): cosine distance between images generated by model A and B under identical noise.
- Right axis (red/orange): DSM test loss on the held-out val split, per model.
- Dotted verticals mark the minima of each curve.

The biased-generalization window is the interval between the blue minimum (cosine distance bottoms out) and the red minimum (test loss bottoms out).

## 1 — Imports and path setup

In [1]:
import os, sys, json, time
from pathlib import Path

# Make the repo importable — this notebook lives in experiments/notebooks/
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent.parent          # .../DiffiT-v2
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))

import numpy as np
import torch
from tqdm.auto import tqdm
import plotly.graph_objects as go
from diffusers.models import AutoencoderKL

# Functions from the analysis script
from analyze_sample_split import (
    match_checkpoints,
    load_model,
    cosine_distance,
    read_test_loss,
)
from diffit import create_diffusion, diffusion_defaults

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

/home/david/anaconda3/envs/diffit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT: /home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2
torch 2.11.0+cu130, CUDA available: True


## 2 — Configuration

Adjust `RUN_A` / `RUN_B` to point at your split-A and split-B run directories. The directory listing auto-detects which run is which based on the `splitA`/`splitB` suffix.

In [2]:
# RUNS_DIR = REPO_ROOT / 'experiments' / 'runs' / '256'
RUNS_DIR = REPO_ROOT / 'experiments' / 'runs' / '512'

# Auto-detect A/B from dirname suffix.
run_dirs = {d.name: d for d in RUNS_DIR.iterdir() if d.is_dir()}
RUN_A = next((str(p) for n, p in run_dirs.items() if 'splitA' in n), None)
RUN_B = next((str(p) for n, p in run_dirs.items() if 'splitB' in n), None)
assert RUN_A and RUN_B, f'Could not find splitA/splitB in {RUNS_DIR}: {list(run_dirs.keys())}'

# --- Analysis knobs --------------------------------------------------------
NUM_SAMPLES     = 64      # images per checkpoint for cosine distance
BATCH_SIZE      = 16
NUM_STEPS       = 50      # DDPM sampling steps per image
MAX_CHECKPOINTS = 20      # set to None for all; otherwise evenly subsampled
BASE_SEED       = 0

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f'RUN_A: {RUN_A}')
print(f'RUN_B: {RUN_B}')
print(f'device: {DEVICE}, amp: {AMP_DTYPE}')

RUN_A: /home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2/experiments/runs/512/00001-diffit-512-splitA-batch128
RUN_B: /home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2/experiments/runs/512/00000-diffit-512-splitB-batch128
device: cuda, amp: torch.bfloat16


## 3 — Discover matching checkpoints and read test loss

In [3]:
pairs = match_checkpoints(RUN_A, RUN_B)
print(f'Found {len(pairs)} matching checkpoint pairs')

# Optional subsample for a faster first pass.
if MAX_CHECKPOINTS is not None and len(pairs) > MAX_CHECKPOINTS:
    idx = np.linspace(0, len(pairs) - 1, MAX_CHECKPOINTS).round().astype(int)
    pairs = [pairs[i] for i in idx]
    print(f'Subsampled to {len(pairs)} checkpoints evenly across the run')

print('kimgs to analyze:', [k for k, _, _ in pairs])

test_a = read_test_loss(RUN_A)
test_b = read_test_loss(RUN_B)
print(f'Test-loss scalars: A={len(test_a)} points, B={len(test_b)} points')

Found 49 matching checkpoint pairs
Subsampled to 20 checkpoints evenly across the run
kimgs to analyze: [409, 1638, 2457, 3686, 4505, 5734, 6553, 7782, 8601, 9830, 10649, 11878, 12697, 13926, 14745, 15974, 16793, 18022, 18841, 20000]
Test-loss scalars: A=49 points, B=49 points


## 4 — Load the VAE decoder (once)

In [4]:
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-ema').to(DEVICE).eval()
for p in vae.parameters():
    p.requires_grad_(False)

diffusion = create_diffusion(**diffusion_defaults())
print('VAE + diffusion schedule ready.')

/home/david/anaconda3/envs/diffit/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


VAE + diffusion schedule ready.


## 5 — Run the sample-split analysis

For each `(A_i, B_i)` checkpoint pair:
1. Load both EMA models to `DEVICE`.
2. Draw `NUM_SAMPLES` matched-noise image pairs (identical `z_T`, identical per-step noise, identical class labels).
3. Decode via the VAE and measure mean cosine distance over the batch.
4. Free GPU memory and move on.

This is the bulk of the wall-clock time; expect **~30–60 s per checkpoint at NUM_SAMPLES=64, NUM_STEPS=50 on an H200**. Scale accordingly for weaker GPUs.

In [5]:
results = []
for kimg, path_a, path_b in tqdm(pairs, desc='checkpoints'):
    t0 = time.time()

    model_a, image_size, num_classes_a = load_model(path_a, DEVICE)
    model_b, _, num_classes_b          = load_model(path_b, DEVICE)
    assert num_classes_a == num_classes_b, f'Class count mismatch: {num_classes_a} vs {num_classes_b}'
    latent_size = image_size // 8

    mean, sem = cosine_distance(
        model_a, model_b, vae, diffusion, DEVICE,
        num_samples=NUM_SAMPLES,
        latent_size=latent_size,
        batch_size=BATCH_SIZE,
        num_steps=NUM_STEPS,
        num_classes=num_classes_a,
        base_seed=BASE_SEED,
    )

    results.append({
        'kimg': kimg,
        'cosine_distance': mean,
        'cosine_distance_sem': sem,
        'test_loss_a': test_a.get(kimg, float('nan')),
        'test_loss_b': test_b.get(kimg, float('nan')),
    })

    # Free GPU before next iteration
    del model_a, model_b
    torch.cuda.empty_cache()

    print(f'  kimg={kimg:>6d}  cos={mean:.4f}±{sem:.4f}  '
          f'tl_A={results[-1]["test_loss_a"]:.4f}  '
          f'tl_B={results[-1]["test_loss_b"]:.4f}  '
          f'({time.time() - t0:.1f}s)')

print(f'\nDone: {len(results)} checkpoint pairs analyzed.')

checkpoints:   5%|▌         | 1/20 [01:07<21:22, 67.49s/it]

  kimg=   409  cos=0.0030±0.0000  tl_A=0.8703  tl_B=0.8755  (67.5s)


checkpoints:  10%|█         | 2/20 [02:14<20:09, 67.18s/it]

  kimg=  1638  cos=0.0153±0.0002  tl_A=0.2144  tl_B=0.2036  (67.0s)


checkpoints:  15%|█▌        | 3/20 [03:21<19:01, 67.14s/it]

...

  kimg= 18022  cos=0.2243±0.0162  tl_A=0.0781  tl_B=0.0788  (67.2s)


checkpoints:  95%|█████████▌| 19/20 [21:16<01:07, 67.22s/it]

  kimg= 18841  cos=0.2255±0.0165  tl_A=0.0780  tl_B=0.0799  (67.2s)


checkpoints: 100%|██████████| 20/20 [22:23<00:00, 67.20s/it]

  kimg= 20000  cos=0.2161±0.0162  tl_A=0.0913  tl_B=0.0822  (67.2s)

Done: 20 checkpoint pairs analyzed.


In [6]:
results

[{'kimg': 409,
  'cosine_distance': 0.0030366675928235054,
  'cosine_distance_sem': 2.537922499349973e-05,
  'test_loss_a': 0.8702513575553894,
  'test_loss_b': 0.8754587173461914},
 {'kimg': 1638,
  'cosine_distance': 0.015273055993020535,
  'cosine_distance_sem': 0.00018848282255615073,
  'test_loss_a': 0.21443724632263184,
  'test_loss_b': 0.2036142796278},
 {'kimg': 2457,
  'cosine_distance': 0.043476562947034836,
  'cosine_distance_sem': 0.004403310793674185,
  'test_loss_a': 0.1331658810377121,
  'test_loss_b': 0.13619309663772583},
 {'kimg': 3686,
  'cosine_distance': 0.12676698341965675,
  'cosine_distance_sem': 0.018649475250766875,
  'test_loss_a': 0.11638707667589188,
  'test_loss_b': 0.11496930569410324},
 {'kimg': 4505,
  'cosine_distance': 0.14192328602075577,
  'cosine_distance_sem': 0.019224257547714632,
  'test_loss_a': 0.10909205675125122,
  'test_loss_b': 0.11401484161615372},
 {'kimg': 5734,
  'cosine_distance': 0.1373284999281168,
  'cosine_distance_sem': 0.0175555

## 6 — Plot (Plotly, dual Y-axis)

In [7]:
import json, math, pathlib

RESULTS_PATH = pathlib.Path.cwd().parent.parent / 'experiments' / 'analysis' / '512' / 'results.json'
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

def _clean(r):
    return {k: (None if isinstance(v, float) and math.isnan(v) else v)
            for k, v in r.items()}

with open(RESULTS_PATH, 'w') as f:
    json.dump([_clean(r) for r in results], f, indent=2)

print(f'Saved {len(results)} records → {RESULTS_PATH}')


Saved 20 records → /home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2/experiments/analysis/512/results.json


In [8]:
import json, pathlib
RESULTS_PATH = pathlib.Path.cwd().parent.parent / 'experiments' / 'analysis' / '512' / 'results.json'
with open(RESULTS_PATH) as f:
    results = json.load(f)

# Turn None back into NaN for the plot code
import math
for r in results:
    for k in ('test_loss_a', 'test_loss_b'):
        if r.get(k) is None:
            r[k] = float('nan')

print(f'Loaded {len(results)} records')


Loaded 20 records


In [10]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# ── helpers ────────────────────────────────────────────────────────────
def read_tb_scalar(run_dir, tag):
    ea = EventAccumulator(run_dir, size_guidance={'scalars': 0}); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.empty(0), np.empty(0)
    s = ea.Scalars(tag)
    return (np.fromiter((e.step  for e in s), float),
            np.fromiter((e.value for e in s), float))

def ema(y, a=0.05):
    if len(y) == 0: return y
    out = np.empty_like(y, float); acc = y[0]
    for i, v in enumerate(y):
        acc = a * v + (1 - a) * acc; out[i] = acc
    return out

# ── gather all series ──────────────────────────────────────────────────
RUNS   = {'A': RUN_A, 'B': RUN_B}
tb     = {s: {tag: read_tb_scalar(r, tag) for tag in ('Loss/train', 'Metrics/FID')}
          for s, r in RUNS.items()}

# Full (non-split) DiffiT training run — reference FID curve.
# RUN_FULL                 = str(RUNS_DIR / '00017-diffit-256-gpus2-batch192_full')
RUN_FULL                 = str(RUNS_DIR / '00018-diffit-512-gpus4-batch256')
fid_full_x, fid_full_y   = read_tb_scalar(RUN_FULL, 'Metrics/FID')

kimgs   = np.array([r['kimg']                for r in results])
# Cosine DISTANCE (1 - cos_similarity), as in Garnier-Brun et al. 2026 Fig 1(a).
# 0 = identical images, larger = more divergent. Because A and B share
# initialization (same --seed), the curve starts near 0 and rises through the
# biased phase to a peak, rather than the U-shape of the paper's different-init setup.
cos     = np.array([r['cosine_distance']     for r in results])
cos_err = np.array([r['cosine_distance_sem'] for r in results])
test    = {'A': np.array([r['test_loss_a']   for r in results], float),
           'B': np.array([r['test_loss_b']   for r in results], float)}

# ── style ──────────────────────────────────────────────────────────────
COLOR_COS  = '#1f77b4'
COLOR_FULL = '#2ca02c'
COLORS     = {'A': '#d62728', 'B': '#ff7f0e'}
DOMAIN_R   = 0.85   # leave room on the right for the second right-axis

# ── figure ─────────────────────────────────────────────────────────────
fig = go.Figure()

# left axis (y): cosine distance
fig.add_trace(go.Scatter(
    x=kimgs, y=cos, error_y=dict(type='data', array=cos_err),
    mode='lines+markers', name='cosine distance',
    line=dict(color=COLOR_COS, width=3), marker=dict(size=9),
    yaxis='y',
))

# right-inner axis (y2): train + test loss
for s in ('A', 'B'):
    kx, ky = tb[s]['Loss/train']
    fig.add_trace(go.Scatter(
        x=kx, y=ema(ky), mode='lines',
        name=f'train loss ({s})',
        line=dict(color=COLORS[s], width=1.5), yaxis='y2',
    ))
    fig.add_trace(go.Scatter(
        x=kimgs, y=test[s], mode='lines+markers',
        name=f'test loss ({s})',
        line=dict(color=COLORS[s], width=2, dash='dash'),
        marker=dict(size=6, symbol='x'), yaxis='y2',
    ))

# right-outer axis (y3, log): FID
for s in ('A', 'B'):
    fx, fy = tb[s]['Metrics/FID']
    fig.add_trace(go.Scatter(
        x=fx, y=fy, mode='lines+markers',
        name=f'FID ({s})',
        line=dict(color=COLORS[s], width=2, dash='dot'),
        marker=dict(size=7, symbol='diamond'), yaxis='y3',
    ))

# FID for the full (non-split) DiffiT training run.
if len(fid_full_x):
    fig.add_trace(go.Scatter(
        x=fid_full_x, y=fid_full_y, mode='lines+markers',
        name='FID (full)',
        line=dict(color=COLOR_FULL, width=2, dash='dot'),
        marker=dict(size=7, symbol='diamond'), yaxis='y3',
    ))

# 5 sample markers (stars) — kimg checkpoints chosen to span the phases:
# pre-bias, bias onset, biased peak, post-bias, converged.
# stars_x = [4500, 5800, 8200, 12500, 16000]
# stars_y = [0.080, 0.025, 0.045, 0.210, 0.245]
# fig.add_trace(go.Scatter(
#     x=stars_x, y=stars_y, mode='markers',
#     name='point of interest',
#     marker=dict(symbol='star', size=22, color='#2ecc40',
#                 line=dict(width=1.5, color='#0a5a14')),
#     yaxis='y',
# ))

# vertical markers
# Paper convention: distance MIN = onset of biased phase. With same-init runs,
# distance starts at 0 and PEAKS at biased-phase center → use nanargmax here.
if len(cos) > 1:
    fig.add_vline(x=float(kimgs[np.nanargmax(cos)]),
                  line=dict(color=COLOR_COS, dash='dot', width=1.5))
tl_mean = np.nanmean(np.stack([test['A'], test['B']]), axis=0)
if not np.all(np.isnan(tl_mean)):
    fig.add_vline(x=float(kimgs[np.nanargmin(tl_mean)]),
                  line=dict(color='#666', dash='dot', width=1.5))

# layout — square figure with three y-axes, legend inside the plot
SIDE = 750
fig.update_layout(
    # title='Biased generalization in DiffiT (256²) — cosine distance / losses / FID',
    template='plotly_white',
    hovermode='x unified',
    height=750, width=900,
    xaxis=dict(title='kimg', domain=[0.0, DOMAIN_R]),
    yaxis =dict(title=dict(text='Sample-split cosine distance',
                           font=dict(color=COLOR_COS)),
                tickfont=dict(color=COLOR_COS)),
    yaxis2=dict(title='DSM loss (train / test)',
                overlaying='y', side='right'),
    yaxis3=dict(title='FID (log)',
                overlaying='y', side='right', anchor='free',
                position=DOMAIN_R + 0.10, type='log'),
    legend=dict(orientation='v',
                x=0.02, y=0.98, xanchor='left', yanchor='top',
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='rgba(0,0,0,0.15)', borderwidth=1),
)

fig.show()
